# Configuración

In [1]:
import sys
import os

# Agregar carpeta raíz del proyecto
sys.path.append(
    os.path.abspath("..")
)

# Cargar datos

In [2]:
from src.cargar_datos import cargar_datos

personal, calendario, configuracion_puestos = cargar_datos(
    "../data/cerebro_farallones.xlsx"
)

print("Personal:", len(personal))
print("Fechas calendario:", len(calendario))

Personal: 77
Fechas calendario: 31


# Crear turnos

In [3]:
from src.crear_turnos import crear_turnos

turnos_df = crear_turnos(
    calendario,
    configuracion_puestos
)

display(turnos_df)

,id,puesto,tipo,fecha,fecha_inicio,fecha_fin,duracion_dias,horario,personas,tipo_dia,festivo
0,PP-01,Pato-Pance,turno,2026-09-05,2026-09-05,2026-09-05,1,dia,5,sábado,SI
1,TO-01,Topacio,turno,2026-09-05,2026-09-05,2026-09-05,1,dia,1,sábado,SI
2,PP-02,Pato-Pance,turno,2026-09-06,2026-09-06,2026-09-06,1,dia,5,domingo,SI
3,TO-02,Topacio,turno,2026-09-06,2026-09-06,2026-09-06,1,dia,1,domingo,SI
4,PL-01,Pato-Leonera,bloque,2026-09-07,2026-09-07,2026-09-10,4,24h,4,lunes,NO
5,PL-02,Pato-Leonera,bloque,2026-09-11,2026-09-11,2026-09-14,4,24h,4,viernes,NO
6,PP-03,Pato-Pance,turno,2026-09-12,2026-09-12,2026-09-12,1,dia,5,sábado,SI
7,TO-03,Topacio,turno,2026-09-12,2026-09-12,2026-09-12,1,dia,1,sábado,SI
8,PP-04,Pato-Pance,turno,2026-09-13,2026-09-13,2026-09-13,1,dia,5,domingo,SI
9,TO-04,Topacio,turno,2026-09-13,2026-09-13,2026-09-13,1,dia,1,domingo,SI


# Resumen de puestos

In [4]:
print("\n=== RESUMEN DE TURNOS ===")

print(
    turnos_df.groupby(
        "puesto"
    ).agg(
        turnos=("id", "count"),
        personas=("personas", "sum")
    )
)


=== RESUMEN DE TURNOS ===
              turnos  personas
puesto                        
Amor y Paz         3        12
Pato-Leonera       7        28
Pato-Pance         9        45
Topacio            9         9


# Crear modelo

In [5]:
from src.modelo import construir_modelo_base

model, x = construir_modelo_base(
    personal,
    turnos_df
)

print("Modelo base creado correctamente.")

Modelo base creado correctamente.


# Restricciones específicas

In [6]:
from src.restricciones_bloques import (
    agregar_restricciones_bloques
)

from src.restricciones_puestos import (
    agregar_restricciones_puestos
)


agregar_restricciones_bloques(
    model,
    x,
    personal,
    turnos_df
)


agregar_restricciones_puestos(
    model,
    x,
    personal,
    turnos_df
)

print("Restricciones agregadas correctamente.")

Restricciones agregadas correctamente.


# Función objetivo y balance de carga

In [7]:
from src.objetivo import (
    agregar_funcion_objetivo
)


carga = agregar_funcion_objetivo(

    model=model,

    x=x,

    personal=personal,

    turnos_df=turnos_df,

    semilla=None

)

print("Función objetivo agregada correctamente.")

Función objetivo agregada correctamente.


# Resolver modelo

In [8]:
from ortools.sat.python import cp_model


solver = cp_model.CpSolver()

status = solver.Solve(model)

print(
    "Estado:",
    solver.StatusName(status)
)

Estado: OPTIMAL


# Validar solución

In [10]:
from src.validar_solucion import (
    validar_solucion
)


if status in [
    cp_model.FEASIBLE,
    cp_model.OPTIMAL
]:

    errores = validar_solucion(
        solver,
        x,
        personal,
        turnos_df
    )

else:

    errores = [
        "No se encontró solución."
    ]


VALIDACIÓN DE LA SOLUCIÓN

✓ SOLUCIÓN VÁLIDA
Todas las reglas verificadas se cumplen.


# Reportes

In [11]:
from src.reportes import (
    imprimir_asignaciones,
    crear_resumen_carga
)


if status in [
    cp_model.FEASIBLE,
    cp_model.OPTIMAL
]:

    imprimir_asignaciones(
        solver,
        x,
        personal,
        turnos_df
    )

    resumen_df = crear_resumen_carga(
        solver,
        x,
        personal,
        turnos_df
    )

    display(resumen_df)

else:

    print(
        "No se encontró solución."
    )


=== ASIGNACIONES ===

PP-01 | 2026-09-05 - 2026-09-05 | Pato-Pance | dia
- Alexander Gomez
- Danny Leandro Mora
- Joseph Emerson Lemos Torres
- Wilfrido Ibarbo
- Viviana urbano

TO-01 | 2026-09-05 - 2026-09-05 | Topacio | dia
- Miguel Castro

PP-02 | 2026-09-06 - 2026-09-06 | Pato-Pance | dia
- Alvaro Libreros
- Belisario Solis
- Cesar Rosasco
- Hernan Montoya
- Marino Lasso

TO-02 | 2026-09-06 - 2026-09-06 | Topacio | dia
- Maria Fernanda Parra

PL-01 | 2026-09-07 - 2026-09-10 | Pato-Leonera | 24h
- Alejandra Garcia
- Camilo Castañeda
- Guillermo Pantoja
- Rubiela Pechene Figueroa

PL-02 | 2026-09-11 - 2026-09-14 | Pato-Leonera | 24h
- Alexander Gomez
- Diana Murillo
- Karen Osorio
- Libardo Torres

PP-03 | 2026-09-12 - 2026-09-12 | Pato-Pance | dia
- Andres Moreno
- Guillermo Pantoja
- Paola Alzate
- Samuel Barona
- Sofia Martinez

TO-03 | 2026-09-12 - 2026-09-12 | Topacio | dia
- Felipe Garcia

PP-04 | 2026-09-13 - 2026-09-13 | Pato-Pance | dia
- Alexander Morales
- Alfredo Abadia


,nombre,cargo,estrategia,activo,conductor,amor_y_paz,pato_leonera,pato_pance,topacio,total_puestos
0,Alfredo Abadia,OPERARIO,FUNCIONARIO,SI,CARRO y MOTO,0,4,2,0,6
1,Rafael Pardo,OPERARIO,ECOTURISMO,SI,MOTO,4,0,1,0,5
2,Wilfrido Ibarbo,OPERARIO,FUNCIONARIO,SI,NO,4,0,1,0,5
3,Jesus David Caicedo,TECNICO,PVC,SI,NO,0,4,1,0,5
4,Juan Manuel Guzman,TECNICO,FUNCIONARIO,SI,CARRO,0,4,1,0,5
...,...,...,...,...,...,...,...,...,...,...
72,Maria Fernanda Parra,PROFESIONAL,RESTAURACION,SI,NO,0,0,0,1,1
73,Gustavo Rodríguez,PROFESIONAL,MONITOREO,SI,NO,0,0,0,1,1
74,Danny Leandro Mora,TECNICO,MINERIA,SI,NO,0,0,1,0,1
75,Joseph Emerson Lemos Torres,PROFESIONAL,PVC,SI,NO,0,0,1,0,1


# Resumen por cargo

In [12]:
if status in [
    cp_model.FEASIBLE,
    cp_model.OPTIMAL
]:

    resumen_cargo = (
        resumen_df
        .groupby("cargo")
        .agg(
            personas=("nombre", "count"),
            puestos_totales=("total_puestos", "sum"),
            promedio_puestos=(
                "total_puestos",
                "mean"
            ),
            minimo=("total_puestos", "min"),
            maximo=("total_puestos", "max")
        )
        .round(2)
    )

    display(resumen_cargo)

,personas,puestos_totales,promedio_puestos,minimo,maximo
cargo,,,,,
OPERARIO,18,55,3.06,1,6
PROFESIONAL,31,66,2.13,0,4
TECNICO,21,76,3.62,1,5
TECNOLOGO,7,17,2.43,1,4


# Exportar Excel

In [13]:
from src.exportar_resultados import (
    exportar_excel
)


if status in [
    cp_model.FEASIBLE,
    cp_model.OPTIMAL
] and len(errores) == 0:

    exportar_excel(
        solver=solver,
        x=x,
        personal=personal,
        turnos_df=turnos_df,
        archivo_salida="../outputs/asignacion_septiembre.xlsx"
    )

    print(
        "\n✓ Archivo exportado correctamente."
    )


✓ Excel exportado:
../outputs/asignacion_septiembre.xlsx

✓ Archivo exportado correctamente.
